In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# =========================
# LOAD DATA
# =========================

df = pd.read_csv("carbon_emission.csv")

# Clean data
df = df.dropna()
df = df.drop_duplicates()

# =========================
# PREPARE DATA
# =========================

# Top emitters
top_10 = df.sort_values(by="co2_2020", ascending=False).head(10)

# Highest increase
change_data = df.sort_values(by="change_2018_2020", ascending=False).head(10)

# Continent data
continent_data = (
    df.groupby("continent")["co2_2020"]
    .sum()
    .reset_index()
)

# Income group data
income_data = (
    df.groupby("income_group")["co2_2020"]
    .sum()
    .reset_index()
)

# KPI Values
total_emission = round(df["co2_2020"].sum(), 2)
avg_emission = round(df["co2_2020"].mean(), 2)
max_country = top_10.iloc[0]["country_name"]
max_emission = round(top_10.iloc[0]["co2_2020"], 2)

# =========================
# CREATE DASHBOARD LAYOUT
# =========================

fig = make_subplots(
    rows=4,
    cols=2,
    specs=[
        [{"type": "indicator"}, {"type": "indicator"}],
        [{"type": "xy"}, {"type": "xy"}],
        [{"type": "domain"}, {"type": "xy"}],
        [{"type": "xy", "colspan": 2}, None]
    ],
    subplot_titles=(
        "",
        "",
        "Top 10 CO₂ Emitting Countries",
        "CO₂ Emissions Comparison (2018 vs 2020)",
        "CO₂ Emissions by Continent",
        "CO₂ Emissions by Income Group",
        "Emission Change Analysis"
    ),
    vertical_spacing=0.12,
    horizontal_spacing=0.08
)

# =========================
# KPI CARD 1
# =========================

fig.add_trace(
    go.Indicator(
        mode="number",
        value=total_emission,
        title={
            "text": "<b>Total Global CO₂ Emissions (2020)</b>",
            "font": {"size": 22, "color": "white"}
        },
        number={
            "font": {"size": 40, "color": "#00E5FF"},
            "suffix": " Mt"
        }
    ),
    row=1,
    col=1
)

# =========================
# KPI CARD 2
# =========================

fig.add_trace(
    go.Indicator(
        mode="number",
        value=max_emission,
        title={
            "text": f"<b>Highest Emitter: {max_country}</b>",
            "font": {"size": 22, "color": "white"}
        },
        number={
            "font": {"size": 40, "color": "#FF9800"},
            "suffix": " Mt"
        }
    ),
    row=1,
    col=2
)

# =========================
# BAR CHART - TOP EMITTERS
# =========================

fig.add_trace(
    go.Bar(
        x=top_10["country_name"],
        y=top_10["co2_2020"],
        marker=dict(
            color=top_10["co2_2020"],
            colorscale="Turbo"
        ),
        text=top_10["co2_2020"].round(1),
        textposition="outside",
        hovertemplate="<b>%{x}</b><br>CO₂: %{y} Mt<extra></extra>"
    ),
    row=2,
    col=1
)

# =========================
# GROUPED COMPARISON CHART
# =========================

fig.add_trace(
    go.Bar(
        x=top_10["country_name"],
        y=top_10["co2_2018"],
        name="2018",
        marker_color="#00BCD4"
    ),
    row=2,
    col=2
)

fig.add_trace(
    go.Bar(
        x=top_10["country_name"],
        y=top_10["co2_2020"],
        name="2020",
        marker_color="#FF9800"
    ),
    row=2,
    col=2
)

# =========================
# DONUT CHART
# =========================

fig.add_trace(
    go.Pie(
        labels=continent_data["continent"],
        values=continent_data["co2_2020"],
        hole=0.55,
        textinfo="percent+label",
        marker=dict(colors=px.colors.qualitative.Bold)
    ),
    row=3,
    col=1
)

# =========================
# INCOME GROUP ANALYSIS
# =========================

fig.add_trace(
    go.Bar(
        x=income_data["income_group"],
        y=income_data["co2_2020"],
        marker=dict(
            color=income_data["co2_2020"],
            colorscale="Viridis"
        ),
        text=income_data["co2_2020"].round(1),
        textposition="outside"
    ),
    row=3,
    col=2
)

# =========================
# SCATTER ANALYSIS
# =========================

fig.add_trace(
    go.Scatter(
        x=df["co2_2018"],
        y=df["co2_2020"],
        mode="markers",
        marker=dict(
            size=10,
            color=df["pct_change_2018_2020"],
            colorscale="Plasma",
            showscale=True,
            colorbar=dict(title="% Change")
        ),
        text=df["country_name"],
        hovertemplate=(
            "<b>%{text}</b><br>"
            "2018: %{x} Mt<br>"
            "2020: %{y} Mt<extra></extra>"
        )
    ),
    row=4,
    col=1
)

# =========================
# DASHBOARD STYLING
# =========================

fig.update_layout(
    title={
        "text": "🌍 GLOBAL CARBON EMISSIONS ANALYTICS DASHBOARD",
        "x": 0.5,
        "font": {
            "size": 30,
            "color": "white",
            "family": "Arial Black"
        }
    },
    template="plotly_dark",
    height=1400,
    width=1600,
    paper_bgcolor="#0D1117",
    plot_bgcolor="#161B22",
    font=dict(color="white"),
    barmode="group",
    hoverlabel=dict(
        bgcolor="black",
        font_size=14
    ),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

# =========================
# AXIS CUSTOMIZATION
# =========================

fig.update_xaxes(showgrid=False)
fig.update_yaxes(gridcolor="rgba(255,255,255,0.1)")

# =========================
# SHOW DASHBOARD
# =========================

fig.show()
fig.write_image("carbon_dashboard.png", width=1920, height=1080, scale=2)
